In [1]:
import numpy as np

# import matplotlib.ticker as ticker
# import matplotlib.pylab as plt

# import aplpy
# import pylab as pl

# import astropy.io.fits as fits
# from astropy.wcs import WCS
from astropy import units as u

from spectral_cube import SpectralCube

from astropy.coordinates import SkyCoord
from astropy.wcs.utils import skycoord_to_pixel
# from astropy.wcs.utils import pixel_to_skycoord

from matplotlib.patches import Ellipse

# from astropy.stats import mad_std

# import matplotlib.pyplot as plt
# from skimage.morphology import remove_small_objects
from astropy.wcs.utils import proj_plane_pixel_scales

import matplotlib as mpl

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman"],  # or "Computer Modern"
    "font.size": 14,

    "axes.labelsize": 14,
    "axes.titlesize": 16,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12,

    "mathtext.fontset": "cm",  # Computer Modern for LaTeX math
})

In [2]:
def plot_contours(data_array, color, sigma, sigma_min, sigma_max, sigma_step, ax, extent):

    sigma_levels = np.arange(sigma_min, sigma_max + sigma_step, sigma_step)
    levels = sigma * sigma_levels

    return ax.contour(
        data_array,
        levels=levels,
        colors=color,
        linewidths=0.5,
        origin='lower',
        extent=extent,
    )

def beam(beam_minor, beam_major, beam_pa, beam_xposition, beam_yposition):
    # Set beam location
    beam_x = ra_offsets.min() + beam_xposition  
    beam_y = dec_offsets.min() + beam_yposition

    # Create the beam ellipse
    return Ellipse(
        (beam_x, beam_y),
        width=beam_major,
        height=beam_minor,
        angle=beam_pa,
        edgecolor='black',
        facecolor='black',
        alpha=0.7,
        zorder=10
    )


In [ ]:
extent=[
            ra_offsets.min(), ra_offsets.max(),
            dec_offsets.min(), dec_offsets.max()
        ]

# Finding Systemic Velocity

In [ ]:
fits_files = {
    "H2CO": "/Users/ivarismartinez/Desktop/Research/REU23/ALMA/HOPS164/H2CO/HOPS164_H2CO_Tp12m7m_Combine_pbcor_masked.fits",
    "C18O": "/Users/ivarismartinez/Desktop/Research/REU23/ALMA/HOPS164/C18O/HOPS164_C18O_Tp12m7m_Combine_pbcor_masked.fits",}

source_coord = SkyCoord("5h37m00.425s", "-6d37m10.89s", frame='icrs')

In [ ]:
v_peaks = {}

for name, file in fits_files.items():

    cube = SpectralCube.read(file)
    cube = cube.with_spectral_unit(u.km/u.s, velocity_convention='radio')

    wcs = cube.wcs

    # --- Get pixel position ---
    x, y = skycoord_to_pixel(source_coord, wcs.celestial)
    x, y = int(x), int(y)

    # --- Pixel scale ---
    pixscale = proj_plane_pixel_scales(wcs.celestial)[0] * 3600  # arcsec/pixel

    radius_arcsec = 1.0
    radius_pix = radius_arcsec / pixscale

    # --- Build circular mask ---
    ny, nx = cube.shape[1], cube.shape[2]
    yy, xx = np.ogrid[:ny, :nx]
    mask = (xx - x)**2 + (yy - y)**2 <= radius_pix**2

    # --- Extract spectrum ---
    spectrum = np.array([
        np.nanmean(channel[mask].value)
        for channel in cube
    ])

    vel_axis = cube.spectral_axis.value

    # centroid (NOT max!)
    v_peak = np.nansum(vel_axis * spectrum) / np.nansum(spectrum)

    v_peaks[name] = v_peak

    print(f"{name}: v_centroid = {v_peak:.2f} km/s")

# --- Systemic velocity ---
good_tracers = ["N2Dp", "C18O"]

velocities = np.array([v_peaks[m] for m in good_tracers])

v_sys = np.mean(velocities)
v_std = np.std(velocities)

print("\n--- Systemic velocity ---")
print(f"v_sys = {v_sys:.2f} ± {v_std:.2f} km/s")